# Notebook 03 — Avaliação de Performance e Latência

## Aula 01: Fundamentos de Latência e Performance em ML

---

### 🎯 Objetivos deste Notebook

1. Avaliar qualidade dos modelos treinados (accuracy, F1, AUC-ROC)
2. Analisar o **trade-off entre qualidade e latência**
3. Gerar visualizações comparativas (confusion matrix, ROC curves)
4. Criar ranking final de modelos considerando qualidade + performance
5. Documentar conclusões e recomendações

### 📖 Conceito Teórico

Em produção, a escolha do modelo não depende apenas da acurácia.
O **trade-off qualidade vs. latência** é fundamental:

> Um modelo 1% mais preciso mas 10x mais lento pode ser inaceitável
> se a aplicação tem SLA de latência < 50ms (Dean & Barroso, 2013).

O **Pareto frontier** entre qualidade e latência mostra as opções ótimas:
modelos onde é impossível melhorar uma métrica sem piorar a outra.

---

In [ ]:
# Imports necessários
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório raiz ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation import (
    benchmark_batch_sizes,
    calculate_latency_metrics,
    calculate_metrics,
    compare_models_latency,
    plot_batch_size_analysis,
    plot_confusion_matrix,
    plot_latency_comparison,
    plot_roc_curve,
)
from src.utils import load_metrics, load_model, save_metrics, setup_output_dirs

# Configurações
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

dirs = setup_output_dirs(str(Path.cwd().parent))
print("Ambiente configurado com sucesso!")

## 1. Carregar Dados e Modelos

Carregamos os dados de teste e os modelos treinados no Notebook 02.

In [ ]:
# Carregar dados de teste
processed_dir = Path.cwd().parent / "data" / "processed"
X_test = np.load(str(processed_dir / "X_test.npy"))
y_test = np.load(str(processed_dir / "y_test.npy"))

print(f"X_test: {X_test.shape}  |  y_test: {y_test.shape}")

# Carregar modelos treinados
model_dir = dirs["models"]
model_names = ["logistic_regression", "random_forest", "gradient_boosting", "svm"]
models = {}

for name in model_names:
    model_path = str(model_dir / f"{name}.pkl")
    if Path(model_path).exists():
        models[name] = load_model(model_path)
        print(f"  Modelo '{name}' carregado.")
    else:
        print(f"  AVISO: Modelo '{name}' não encontrado. Execute o Notebook 02 primeiro.")

print(f"\n{len(models)} modelos carregados.")

## 2. Avaliação de Qualidade dos Modelos

Calculamos as métricas de classificação padrão para cada modelo
no conjunto de teste.

In [ ]:
# Calcular métricas de qualidade
quality_results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    metrics = calculate_metrics(y_test, y_pred)
    metrics["model"] = name
    quality_results.append(metrics)

quality_df = pd.DataFrame(quality_results)
quality_df = quality_df[["model", "accuracy", "precision", "recall", "f1_score"]]

print("=" * 70)
print("MÉTRICAS DE QUALIDADE NO CONJUNTO DE TESTE")
print("=" * 70)
print(quality_df.round(4).to_string(index=False))

In [ ]:
# Visualização comparativa das métricas de qualidade
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(quality_df))
width = 0.2
metrics_to_plot = ["accuracy", "precision", "recall", "f1_score"]
colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax.bar(x + i * width, quality_df[metric], width, label=metric.replace("_", " ").title(),
           color=color, alpha=0.8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(quality_df["model"], rotation=45, ha="right")
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.set_title("Métricas de Qualidade por Modelo", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig.savefig(str(dirs["figures"] / "03_quality_metrics.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Matriz de Confusão por Modelo

A matriz de confusão permite analisar os tipos de erro de cada modelo,
o que pode influenciar a escolha em cenários onde falsos positivos
e falsos negativos têm custos diferentes.

In [ ]:
# Matrices de confusão para todos os modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    from sklearn.metrics import confusion_matrix as cm_func
    cm = cm_func(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Classe 0", "Classe 1"],
                yticklabels=["Classe 0", "Classe 1"],
                ax=axes[idx])
    axes[idx].set_title(f"{name}", fontsize=12)
    axes[idx].set_xlabel("Predito")
    axes[idx].set_ylabel("Real")

fig.suptitle("Matrizes de Confusão — Conjunto de Teste", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(str(dirs["figures"] / "03_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Benchmark de Latência de Inferência

Conforme o protocolo MLPerf (Reddi et al., 2020), realizamos múltiplas
iterações de inferência com warm-up para capturar a distribuição real.

In [ ]:
# Benchmark de latência para todos os modelos
latency_results = []

for name, model in models.items():
    print(f"Benchmarking {name}...")
    bench = calculate_latency_metrics(model, X_test, n_iterations=200)
    latency_results.append(bench)

latency_df = pd.DataFrame(latency_results)

print("\n" + "=" * 80)
print("BENCHMARK DE LATÊNCIA DE INFERÊNCIA (ms) — Conjunto de Teste")
print("=" * 80)
display_cols = ["model", "mean_ms", "std_ms", "p50_ms", "p95_ms", "p99_ms",
                "throughput_samples_per_sec"]
print(latency_df[display_cols].round(4).to_string(index=False))

In [ ]:
# Visualização da comparação de latência
fig = plot_latency_comparison(
    latency_df,
    save_path=str(dirs["figures"] / "03_latency_comparison.png")
)
plt.show()

## 5. Trade-off Qualidade vs. Latência

A análise mais importante: combinar qualidade e performance em uma
única visão. O **Pareto frontier** mostra os modelos ótimos —
onde melhorar qualidade requer piorar latência e vice-versa.

> Em produção, nem sempre o modelo mais preciso é o melhor.
> O SLA de latência frequentemente define qual modelo pode ser usado.

In [ ]:
# Combinar qualidade e latência
combined_df = quality_df.merge(
    latency_df[["model", "mean_ms", "p95_ms", "throughput_samples_per_sec"]],
    on="model"
)

print("=" * 80)
print("RANKING COMBINADO — QUALIDADE + PERFORMANCE")
print("=" * 80)
print(combined_df.round(4).to_string(index=False))

In [ ]:
# Gráfico de Pareto: F1-Score vs Latência
fig, ax = plt.subplots(figsize=(10, 7))

colors_map = {
    "logistic_regression": "#3498db",
    "random_forest": "#2ecc71",
    "gradient_boosting": "#e74c3c",
    "svm": "#f39c12",
}

for _, row in combined_df.iterrows():
    color = colors_map.get(row["model"], "gray")
    ax.scatter(row["mean_ms"], row["f1_score"], s=200, c=color,
              edgecolors="black", linewidth=1.5, zorder=5)
    ax.annotate(row["model"], (row["mean_ms"], row["f1_score"]),
               textcoords="offset points", xytext=(10, 10), fontsize=10,
               arrowprops=dict(arrowstyle="->", color="gray", lw=0.5))

ax.set_xlabel("Latência Média de Inferência (ms)", fontsize=12)
ax.set_ylabel("F1-Score", fontsize=12)
ax.set_title("Trade-off Qualidade vs. Latência\n(Ideal: canto superior esquerdo)",
             fontsize=14)
ax.grid(True, alpha=0.3)

# Adicionar zona de SLA (exemplo: < 10ms)
ax.axvline(x=10, color="red", linestyle="--", alpha=0.5, label="SLA = 10 ms")
ax.legend(fontsize=10)

plt.tight_layout()
fig.savefig(str(dirs["figures"] / "03_quality_vs_latency.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Análise de Batch Size para Melhor Modelo

Para o modelo com melhor trade-off, analisamos em detalhe o impacto
do batch size, que é crucial para cenários de processamento em lote.

In [ ]:
# Identificar melhor modelo (considerando F1 > threshold E latência razoável)
best_model_name = combined_df.sort_values("f1_score", ascending=False).iloc[0]["model"]
best_model = models[best_model_name]
print(f"Melhor modelo para análise detalhada: {best_model_name}")

# Análise de batch size
batch_sizes = [1, 5, 10, 25, 50, 100, 200, 500, 1000]
batch_df = benchmark_batch_sizes(
    best_model, X_test, batch_sizes=batch_sizes, n_iterations=100
)

fig = plot_batch_size_analysis(
    batch_df,
    save_path=str(dirs["figures"] / "03_batch_size_analysis.png")
)
plt.show()

print(batch_df.round(4).to_string(index=False))

## 7. Relatório Final Comparativo

Consolidação de todos os resultados em uma tabela final com
recomendações para diferentes cenários de deployment.

In [ ]:
# Tabela final resumo
print("=" * 90)
print("RELATÓRIO FINAL — PERFORMANCE DE MODELOS")
print("=" * 90)
print()

# Ranking por cenário
ranking_latency = combined_df.sort_values("mean_ms").reset_index(drop=True)
ranking_quality = combined_df.sort_values("f1_score", ascending=False).reset_index(drop=True)
ranking_throughput = combined_df.sort_values("throughput_samples_per_sec",
                                             ascending=False).reset_index(drop=True)

print("📊 RANKING POR CENÁRIO:")
print()
print(f"  🏆 Menor Latência:      {ranking_latency.iloc[0]['model']} "
      f"({ranking_latency.iloc[0]['mean_ms']:.2f} ms)")
print(f"  🏆 Melhor Qualidade:    {ranking_quality.iloc[0]['model']} "
      f"(F1 = {ranking_quality.iloc[0]['f1_score']:.4f})")
print(f"  🏆 Maior Throughput:    {ranking_throughput.iloc[0]['model']} "
      f"({ranking_throughput.iloc[0]['throughput_samples_per_sec']:.0f} amostras/s)")
print()

print("📋 TABELA CONSOLIDADA:")
print(combined_df.round(4).to_string(index=False))

print()
print("💡 RECOMENDAÇÕES:")
print("  - SLA < 5ms (real-time): Logistic Regression")
print("  - SLA < 50ms (web apps): Random Forest ou Gradient Boosting")
print("  - Batch processing (sem SLA): Escolher por qualidade")

In [ ]:
# Salvar relatório final
final_report = {
    "quality_metrics": quality_results,
    "latency_metrics": latency_results,
    "combined_ranking": combined_df.to_dict(orient="records"),
    "recommendations": {
        "real_time_sla_5ms": ranking_latency.iloc[0]["model"],
        "web_app_sla_50ms": ranking_quality.iloc[0]["model"],
        "batch_processing": ranking_quality.iloc[0]["model"],
    },
}

save_metrics(final_report, str(dirs["logs"] / "final_report.json"))
print("Relatório final salvo em outputs/logs/final_report.json")

## 📋 Resumo da Aula

### Principais Aprendizados:

1. **Latência é multidimensional**: Não é apenas sobre inferência — I/O, pré-processamento
   e serialização contribuem significativamente para $L_{total}$

2. **Percentis importam**: A média é insuficiente — P95 e P99 revelam o comportamento
   real em produção (Dean & Barroso, 2013)

3. **Complexidade computacional se reflete em latência**: A teoria (O(d) vs O(t·log n))
   se confirma empiricamente

4. **Batch size é um dial de controle**: Permite ajustar o trade-off
   latência-throughput conforme o cenário

5. **Trade-off qualidade vs. latência**: Nem sempre o melhor modelo em acurácia
   é viável em produção — o SLA de latência pode ser o fator limitante

### 📚 Referências Completas

- Crankshaw, D. et al. (2017). *Clipper: A Low-Latency Online Prediction Serving System*. NSDI '17.
- Dean, J. & Barroso, L. A. (2013). *The Tail at Scale*. CACM, 56(2), 74-80.
- Reddi, V. J. et al. (2020). *MLPerf Inference Benchmark*. ISCA '20.
- Sculley, D. et al. (2015). *Hidden Technical Debt in ML Systems*. NeurIPS '15.
- Li, M. et al. (2014). *Scaling Distributed Machine Learning*. OSDI '14.